This notebook helps parse Quantrocket API specs into MCP resources for the strategy coder agent to use.

## Parse zipline API

In [ ]:
import os
from bs4 import BeautifulSoup
from pathlib import Path

html = Path("zipline_api.html").read_text(encoding='utf-8')
soup = BeautifulSoup(html, "html.parser")

In [ ]:
import json
import re

results = {}

for section in soup.find_all("section"):
    # Check if section starts with h2 block
    if section.find("h2", recursive=False):
        # Add section id as new key in results
        results[section['id']] = {}
    elif section.find("h3", recursive=False):
        # Add section as new key in latest section in results
        parent_section = section.find_parent("section")
        if parent_section and parent_section['id'] in results:
            # Add subsection id as new key in parent section and remainder of the h3 section as value
            results[parent_section['id']][section['id']] = section
    else:
        continue

# Modify keys to be in format of docs/zipline/category/object
new_results = {}
for section_id, subsections in results.items():
    for subsection_id, content in list(subsections.items()):
        new_key = f"docs/pipeline/{section_id}/{subsection_id.split('-', 2)[-1]}"
        new_results[new_key] = content

# Split into all elements
for resource, contents in new_results.items():
    elements = []
    dl = contents.find('dl')
    if dl['class'][1] == 'function':
        new_results[resource] = {
            "type": "function",
            "method_signature": dl.find('dt').text.strip("\u00b6").strip("\u00b6\n"),
            # "description": " ".join(dl.find('dd').find('p').text.split()),
        }
    else:
        new_results[resource] = {
            "type": "class",
            "method_signature": dl.find('dt').text.strip("\u00b6").strip("\u00b6\n").strip("class "),
            # "description": " ".join(dl.find('dd').find('p').text.split()),
        }
    # Add remaining children as a key value pair
    counter = 0
    for child in dl.find('dd').find_all(recursive=False):
        key = counter
        counter += 1
        value = child.text.strip()
        value = re.sub(r"\n +", " ", value)
        new_results[resource][key] = str(value)

# Remove usage guides and see alsos
for resource, metadata in new_results.items():
    # Iterate through values of metadata and remove any that start with "Usage Guide" and the one following it
    keys_to_remove = []
    for key, value in metadata.items():
        if value.startswith("Usage Guide:"):
            keys_to_remove.append(key)
            # Also remove the next key if it exists
            next_key = key + 1
            if next_key in metadata:
                keys_to_remove.append(next_key)
        if value.startswith("See also"):
            keys_to_remove.append(key)
    for key in keys_to_remove:
        del new_results[resource][key]

# Extract out parameters, returns and raises
for resource, metadata in new_results.items():
    parameters = {}
    for key, value in list(metadata.items()):
        if value.startswith("Parameters"):
            if metadata["type"] == "class":
                value = value.replace("Parameters:", "").strip()
                param_list = value.split("\n\n\n")
                for param in param_list:
                    if "\u2013" in param:
                        param_name, description = param.split("\u2013", 1)
                        parameters[param_name.strip()] = ' '.join(description.split())
            elif metadata["type"] == "function":
                parameters_section, others = value.split("Returns:", 1) if "Returns:" in value else (value, "")
                returns_section, raises_section = others.split("Raises:", 1) if "Raises:" in others else (others, "")

                param_list = parameters_section.split("\n\n")
                for param in param_list:
                    if "\u2013" in param:
                        param_name, description = param.split("\u2013", 1)
                        parameters[param_name.strip()] = ' '.join(description.split())
                
                returns = returns_section.strip()
                new_results[resource]["returns"] = ' '.join(returns.split())

                raises = raises_section.strip()
                new_results[resource]["raises"] = ' '.join(raises.split())

            del new_results[resource][key]
    new_results[resource]["parameters"] = parameters

# For classes, extract out attributes and methods
for resource, metadata in new_results.items():
    if metadata["type"] == "class":
        attributes = {}
        methods = {}
        for key, value in list(metadata.items()):
            if isinstance(key, int):
                if ")\u00b6\n\n" in value:
                    # This key is a method
                    method_signature, description = value.split("\u00b6\n\n", 1)
                    methods[method_signature.strip()] = ' '.join(description.split())
                    del new_results[resource][key]
                elif "\u00b6\n\n" in value:
                    # This key is an attribute
                    attribute_name, description = value.split("\u00b6\n\n", 1)
                    attributes[attribute_name.strip()] = ' '.join(description.split())
                    del new_results[resource][key]

        new_results[resource]["attributes"] = attributes
        new_results[resource]["methods"] = methods

# Extract notes and examples
# Notes begin after a value starts with "Notes", and subsequent values that do not start with "Example"
# Examples begin when a value starts with "Examples", and subsequent values that do not start with "Notes"
for resource, metadata in new_results.items():
    notes = []
    examples = {}
    current_section = None
    is_example = False
    for key, value in list(metadata.items()):
        if isinstance(key, int):
            if value == ("Notes"):
                current_section = "notes"
                del new_results[resource][key]
            elif value == ("Examples"):
                current_section = "examples"
                del new_results[resource][key]
                is_example = False
            else:
                if current_section == "notes":
                    notes.append(' '.join(value.split()))
                    del new_results[resource][key]
                elif current_section == "examples":
                    if not is_example:
                        examples[value.strip()] = metadata[key + 1].strip()
                        is_example = True
                    else:
                        is_example = False
                    del new_results[resource][key]
    new_results[resource]["notes"] = notes
    new_results[resource]["examples"] = examples

# Merge remaining integer keys into description field
for resource, metadata in new_results.items():
    descriptions = []
    for key, value in list(metadata.items()):
        if isinstance(key, int):
            descriptions.append(value)
            del new_results[resource][key]
    new_results[resource]["description"] = ' '.join(descriptions)


with open("zipline_api_parsed.json", "w", encoding='utf-8') as f:
    json.dump(new_results, f, indent=2)

## Parse pipeline api

In [98]:
import os
from bs4 import BeautifulSoup
from pathlib import Path

html = Path("pipeline_api.html").read_text(encoding='utf-8')
soup = BeautifulSoup(html, "html.parser")

In [101]:
import json
import re

results = {}

for section in soup.find_all("section"):
    # Check if section starts with h2 block
    if section.find("h2", recursive=False):
        # Add section id as new key in results
        results[section['id']] = {}
    elif section.find("h3", recursive=False):
        # Add section as new key in latest section in results
        parent_section = section.find_parent("section")
        if parent_section and parent_section['id'] in results:
            # Add subsection id as new key in parent section and remainder of the h3 section as value
            results[parent_section['id']][section['id']] = section
    else:
        continue

# Modify keys to be in format of docs/zipline/category/object
new_results = {}
for section_id, subsections in results.items():
    for subsection_id, content in list(subsections.items()):
        new_key = f"docs/pipeline/{section_id}/{subsection_id.split('-', 2)[-1]}"
        new_results[new_key] = content

# Split into all elements
for resource, contents in new_results.items():
    elements = []
    dl = contents.find('dl')
    if dl['class'][1] == 'function':
        new_results[resource] = {
            "type": "function",
            "method_signature": re.sub(r"\n +", " ", dl.find('dt').text.strip("\u00b6").strip("\u00b6\n")),
            # "description": " ".join(dl.find('dd').find('p').text.split()),
        }
    else:
        new_results[resource] = {
            "type": "class",
            "method_signature": re.sub(r"\n +", " ", dl.find('dt').text.strip("\u00b6").strip("\u00b6\n").strip("class ")),
            # "description": " ".join(dl.find('dd').find('p').text.split()),
        }
    # Add remaining children as a key value pair
    counter = 0
    for child in dl.find('dd').find_all(recursive=False):
        key = counter
        counter += 1
        value = child.text.strip()
        value = re.sub(r"\n +", " ", value)
        new_results[resource][key] = str(value)

# Remove usage guides and see alsos
for resource, metadata in new_results.items():
    # Iterate through values of metadata and remove any that start with "Usage Guide" and the one following it
    keys_to_remove = []
    for key, value in metadata.items():
        if value.startswith("Usage Guide:"):
            keys_to_remove.append(key)
            # Also remove the next key if it exists
            next_key = key + 1
            if next_key in metadata:
                keys_to_remove.append(next_key)
        if value.startswith("See also"):
            keys_to_remove.append(key)
    for key in keys_to_remove:
        del new_results[resource][key]

# Extract out parameters, returns and raises
for resource, metadata in new_results.items():
    parameters = {}
    for key, value in list(metadata.items()):
        if value.startswith("Parameters"):
            if metadata["type"] == "class":
                value = value.replace("Parameters:", "").strip()
                param_list = value.split("\n\n\n")
                for param in param_list:
                    if "\u2013" in param:
                        param_name, description = param.split("\u2013", 1)
                        parameters[param_name.strip()] = ' '.join(description.split())
            elif metadata["type"] == "function":
                parameters_section, others = value.split("Returns:", 1) if "Returns:" in value else (value, "")
                returns_section, raises_section = others.split("Raises:", 1) if "Raises:" in others else (others, "")

                param_list = parameters_section.split("\n\n")
                for param in param_list:
                    if "\u2013" in param:
                        param_name, description = param.split("\u2013", 1)
                        parameters[param_name.strip()] = ' '.join(description.split())
                
                returns = returns_section.strip()
                new_results[resource]["returns"] = ' '.join(returns.split())

                raises = raises_section.strip()
                new_results[resource]["raises"] = ' '.join(raises.split())

            del new_results[resource][key]
    new_results[resource]["parameters"] = parameters

# For classes, extract out attributes and methods
for resource, metadata in new_results.items():
    if metadata["type"] == "class":
        attributes = {}
        methods = {}
        for key, value in list(metadata.items()):
            if isinstance(key, int):
                if ")\u00b6\n\n" in value:
                    # This key is a method
                    method_signature, description = value.split("\u00b6\n\n", 1)
                    methods[method_signature.strip()] = ' '.join(description.split())
                    del new_results[resource][key]
                elif "\u00b6\n\n" in value:
                    # This key is an attribute
                    attribute_name, description = value.split("\u00b6\n\n", 1)
                    attributes[attribute_name.strip()] = ' '.join(description.split())
                    del new_results[resource][key]

        new_results[resource]["attributes"] = attributes
        new_results[resource]["methods"] = methods

# Extract notes and examples
# Notes begin after a value starts with "Notes", and subsequent values that do not start with "Example"
# Examples begin when a value starts with "Examples", and subsequent values that do not start with "Notes"
for resource, metadata in new_results.items():
    notes = []
    examples = ""
    current_section = None
    for key, value in list(metadata.items()):
        if isinstance(key, int):
            if value == ("Notes"):
                current_section = "notes"
                del new_results[resource][key]
            elif value == ("Examples"):
                current_section = "examples"
                del new_results[resource][key]
            else:
                if current_section == "notes":
                    notes.append(' '.join(value.split()))
                    del new_results[resource][key]
                elif current_section == "examples":
                    examples += value.strip() + "\n"
                    del new_results[resource][key]
    new_results[resource]["notes"] = notes
    new_results[resource]["examples"] = examples

# Merge remaining integer keys into description field
for resource, metadata in new_results.items():
    descriptions = []
    for key, value in list(metadata.items()):
        if isinstance(key, int):
            descriptions.append(value)
            del new_results[resource][key]
    new_results[resource]["description"] = ' '.join(descriptions)


with open("pipeline_api_parsed.json", "w", encoding='utf-8') as f:
    json.dump(new_results, f, indent=2)